# 02 — Data Quality and Cleaning

**Objective** — validate the source contract, quantify known issues, and apply explicit cleaning rules.

**Input** — raw analytical sources and join diagnostics.

**Output** — `data/processed/analytical_dataset.csv`.

## 1. Validate the Source Contract

In [1]:
from pathlib import Path
import sys

current_directory = Path.cwd().resolve()
project_root = next(
    directory for directory in [current_directory, *current_directory.parents]
    if (directory / 'src').is_dir() and (directory / 'notebooks').is_dir()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import load_raw_sources, build_analytical_dataset
from src.validation import validate_source_schema, validate_analytical_dataset, clean_analytical_dataset

raw_data_directory = project_root / 'data' / 'raw'
raw_sources = load_raw_sources(raw_data_directory)
validate_source_schema(raw_sources)
analytical_dataset, join_diagnostics = build_analytical_dataset(
    raw_sources['hospitalisations'], raw_sources['examinations']
)
validate_analytical_dataset(analytical_dataset)

{'rows': 2343,
 'duplicate_extra_rows': 5,
 'unknown_year_count': 2,
 'unknown_smoker_count': 2,
 'missing_medical_rows': 8,
 'non_positive_charge_rows': 0}

## 2. Apply Cleaning Rules

In [2]:
cleaned_dataset = clean_analytical_dataset(analytical_dataset)
processed_data_path = project_root / 'data' / 'processed' / 'analytical_dataset.csv'
processed_data_path.parent.mkdir(parents=True, exist_ok=True)
cleaned_dataset.to_csv(processed_data_path, index=False)
cleaned_dataset[['year', 'charges', 'Heart Issues', 'smoker', 'NumberOfMajorSurgeries']].head()

,year,charges,Heart Issues,smoker,NumberOfMajorSurgeries
0,1992.0,563.84,No,No,1.0
1,1992.0,570.62,No,No,1.0
2,1993.0,600.00,No,No,1.0
3,1992.0,604.54,No,No,1.0
4,1998.0,637.26,No,No,1.0


## Takeaways

Unknown markers become missing values, binary labels are standardized, and surgery counts become numeric. Missing medical matches remain transparent and are handled later by train-only preprocessing rather than hidden during cleaning.